# 00 — Colab Setup

**Purpose:** Mount Google Drive, clone the repository, install dependencies,
configure paths, verify the environment, and run the synthetic smoke test.

**Expected outputs:**
- Python / PyTorch / CUDA version printed
- GPU name printed (or CPU fallback noted)
- Repository commit hash printed
- All imports succeed
- Synthetic smoke test completes in < 60 seconds

**Run order:** Execute cells top-to-bottom.  Do not skip Cell 1 (Drive mount).

## Cell 1 — Mount Google Drive

In [ ]:
import os

# Only mount if running in Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Drive mounted at /content/drive")
    IN_COLAB = True
except ImportError:
    print("Not running in Colab — skipping Drive mount")
    IN_COLAB = False

# Verify mount
if IN_COLAB:
    if not os.path.isdir('/content/drive/MyDrive'):
        raise RuntimeError(
            "Drive mount failed or MyDrive not accessible. "
            "Check permissions and re-run this cell."
        )
    print("Drive accessible at /content/drive/MyDrive")

## Cell 2 — Clone or update the repository

In [ ]:
import os
import subprocess

# ── CONFIGURE THESE ─────────────────────────────────────────────────────────
REPO_URL  = "https://github.com/zhangcastle/battery-soc-transfer-benchmark.git"
REPO_DIR  = "/content/battery-soc-transfer-benchmark"
BRANCH    = "main"
# ────────────────────────────────────────────────────────────────────────────

if not os.path.isdir(REPO_DIR):
    print(f"Cloning {REPO_URL} → {REPO_DIR}")
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    print(f"Repository already exists at {REPO_DIR} — pulling latest")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)

print(f"Repository at: {REPO_DIR}")

## Cell 3 — Install dependencies from requirements-colab.txt

In [ ]:
import subprocess

REPO_DIR = "/content/battery-soc-transfer-benchmark"

result = subprocess.run(
    ["pip", "install", "-q", "-r", f"{REPO_DIR}/requirements-colab.txt"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("Dependency installation failed.")
print("Dependencies installed.")

## Cell 4 — Install the benchmark package in editable mode

In [ ]:
import subprocess

REPO_DIR = "/content/battery-soc-transfer-benchmark"

result = subprocess.run(
    ["pip", "install", "-q", "-e", REPO_DIR],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("Package installation failed.")
print("Package installed in editable mode.")

## Cell 5 — Detect GPU availability

In [ ]:
import torch

if torch.cuda.is_available():
    n = torch.cuda.device_count()
    print(f"GPU available: {n} device(s)")
    for i in range(n):
        mem = torch.cuda.get_device_properties(i).total_memory
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  ({mem/1e9:.1f} GB)")
else:
    print("No GPU detected — training will use CPU.")
    print("For faster training: Runtime → Change runtime type → T4 GPU")

## Cell 6 — Environment summary

In [ ]:
import sys
import torch

REPO_DIR = "/content/battery-soc-transfer-benchmark"

# Add src to path in case editable install is not yet active
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")

from battery_benchmark.utils.git_info import get_commit_hash
from battery_benchmark.utils.environment import get_environment_info

info = get_environment_info()
commit = get_commit_hash(REPO_DIR)

print("=" * 55)
print("Environment Summary")
print("=" * 55)
print(f"  Python version  : {sys.version.split()[0]}")
print(f"  PyTorch version : {torch.__version__}")
print(f"  CUDA version    : {info.get('cuda_version', 'N/A')}")
if info.get('gpu_name'):
    print(f"  GPU             : {info['gpu_name']}")
else:
    print("  GPU             : None (CPU only)")
print(f"  Commit hash     : {commit}")
print("=" * 55)

## Cell 7 — Configure paths

In [ ]:
import os
from pathlib import Path

# ── CONFIGURE THESE to match your Drive layout ───────────────────────────────
REPO_ROOT    = Path("/content/battery-soc-transfer-benchmark")
DRIVE_ROOT   = Path("/content/drive/MyDrive/battery_soc_benchmark")
RAW_DATA     = DRIVE_ROOT / "data" / "raw"
PROCESSED    = DRIVE_ROOT / "data" / "processed"
OUTPUT_ROOT  = DRIVE_ROOT / "outputs"
COLAB_CACHE  = Path("/content/battery_soc_cache")
PATHS_CONFIG = REPO_ROOT / "configs" / "paths" / "colab_drive.yaml"
# ─────────────────────────────────────────────────────────────────────────────

# Create directories
for d in [RAW_DATA, PROCESSED, OUTPUT_ROOT, COLAB_CACHE]:
    d.mkdir(parents=True, exist_ok=True)

print("Path configuration:")
print(f"  repo_root    : {REPO_ROOT}")
print(f"  drive_root   : {DRIVE_ROOT}")
print(f"  raw_data     : {RAW_DATA}")
print(f"  processed    : {PROCESSED}")
print(f"  output_root  : {OUTPUT_ROOT}")
print(f"  colab_cache  : {COLAB_CACHE}")

## Cell 8 — Create dataset symlinks

Symlinks allow code to reference `data/raw/` (relative) while the actual
files live in Google Drive (never in the git repository).

In [ ]:
import os
from pathlib import Path

REPO_ROOT  = Path("/content/battery-soc-transfer-benchmark")
DRIVE_ROOT = Path("/content/drive/MyDrive/battery_soc_benchmark")

symlinks = {
    REPO_ROOT / "data" / "raw":       DRIVE_ROOT / "data" / "raw",
    REPO_ROOT / "data" / "processed": DRIVE_ROOT / "data" / "processed",
    REPO_ROOT / "outputs":             DRIVE_ROOT / "outputs",
}

for link_path, target_path in symlinks.items():
    target_path.mkdir(parents=True, exist_ok=True)
    if link_path.is_symlink():
        link_path.unlink()
    elif link_path.exists():
        print(f"  Skipping {link_path} (real directory exists)")
        continue
    link_path.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(target_path, link_path)
    print(f"  Linked: {link_path} → {target_path}")

## Cell 9 — Import test

In [ ]:
# Verify all benchmark sub-packages import correctly
import battery_benchmark
from battery_benchmark.data.synthetic import SyntheticDataset
from battery_benchmark.data.splits import split_cycles
from battery_benchmark.preprocessing.normalization import MinMaxNormalizer
from battery_benchmark.preprocessing.sequence_builder import SequenceBuilder
from battery_benchmark.models.vidal_lstm import VidalLSTM
from battery_benchmark.training.trainer import Trainer
from battery_benchmark.evaluation.metrics import compute_metrics
from battery_benchmark.utils.config import load_config
from battery_benchmark.utils.device import get_device

print(f"battery_benchmark version: {battery_benchmark.__version__}")
print("All imports successful.")

## Cell 10 — Synthetic smoke test

Runs the complete training pipeline on synthetic data.  No real datasets needed.
Expected runtime: **< 60 seconds** on CPU, **< 15 seconds** on GPU.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path("/content/battery-soc-transfer-benchmark")

result = subprocess.run(
    [
        sys.executable,
        str(REPO_ROOT / "scripts" / "train.py"),
        "--config", str(REPO_ROOT / "configs" / "experiments" / "smoke_test.yaml"),
        "--paths",  str(REPO_ROOT / "configs" / "paths" / "colab_drive.yaml"),
    ],
    capture_output=True,
    text=True,
    cwd=str(REPO_ROOT),
)

print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("Smoke test FAILED. See error above.")
print("Smoke test PASSED.")

## Cell 11 — Next steps

Setup complete.  Continue to:

- `01_dataset_inventory.ipynb` — explore available datasets
- `02_prepare_panasonic.ipynb` — prepare Panasonic NCR18650PF dataset
- `10_vidal_case1_baseline.ipynb` — run Vidal Case 1 experiment

To resume a training run after a session disconnects:
```python
# In any notebook or script:
cfg["checkpoint_path"] = str(OUTPUT_ROOT / "your_experiment" / "checkpoints" / "last.pt")
cfg["resume"] = True
runner = ExperimentRunner(cfg, output_dir=OUTPUT_ROOT / "your_experiment")
result = runner.run()
```